# Lesson 30: MLOps: Deploying & Monitoring Models

*ML Academy — companion notebook*

Packaging and versioning, input validation, drift detection with PSI and KS tests, concept drift and retraining, behavioural tests, serving and rollout.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-30.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 3. Packaging and versioning a model

In [1]:
import numpy as np
import pandas as pd
import joblib, json, hashlib, sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(0)
def make_loans(n, shift=0.0, flip=0.0):
    income = rng.normal(50 + shift, 15, n); debt = rng.normal(20, 8, n); age = rng.integers(21, 70, n)
    logit = (-0.08 + flip) * (income - 50) + 0.12 * (debt - 20) - 1.5
    return pd.DataFrame({"income": income, "debt": debt, "age": age}), (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
X_train, y_train = make_loans(5000)
X_test, y_test = make_loans(2000)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
joblib.dump(model, "model_v1.joblib")
meta = {"version": "1.0.0", "features": list(X_train.columns), "test_auc": round(auc, 3),
        "sklearn": sklearn.__version__, "sha256": hashlib.sha256(open("model_v1.joblib", "rb").read()).hexdigest()[:12]}
json.dump(meta, open("model_v1.json", "w"), indent=1)
loaded = joblib.load("model_v1.joblib")
print("saved model", meta["version"], " test AUC", meta["test_auc"], " fingerprint length", len(meta["sha256"]))
print("reloaded model gives identical predictions:", bool(np.allclose(loaded.predict_proba(X_test), model.predict_proba(X_test))))

saved model 1.0.0  test AUC 0.818  fingerprint length 12
reloaded model gives identical predictions: True


## 4. Validating inputs

In [2]:
SCHEMA = {"income": (0, 500), "debt": (0, 200), "age": (18, 100)}
def validate(rows):
    problems = []
    for i, r in enumerate(rows):
        for col, (lo, hi) in SCHEMA.items():
            v = r.get(col)
            if v is None or not isinstance(v, (int, float)) or v != v:
                problems.append((i, col, "missing or not a number"))
            elif not lo <= v <= hi:
                problems.append((i, col, f"{v} outside [{lo}, {hi}]"))
    return problems
incoming = [{"income": 62.0, "debt": 18.5, "age": 34}, {"income": 48.0, "age": 29},
            {"income": -5.0, "debt": 30.0, "age": 41}, {"income": 55.0, "debt": "12", "age": 230}]
for p in validate(incoming):
    print("row", p[0], "·", p[1], "·", p[2])

row 1 · debt · missing or not a number
row 2 · income · -5.0 outside [0, 500]
row 3 · debt · missing or not a number
row 3 · age · 230 outside [18, 100]


## 5. Detecting data drift

In [3]:
from scipy import stats
def psi(expected, actual, bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, edges)[0] / len(expected); a = np.histogram(actual, edges)[0] / len(actual)
    e, a = np.clip(e, 1e-4, None), np.clip(a, 1e-4, None)
    return float(np.sum((a - e) * np.log(a / e)))
X_live, _ = make_loans(2000, shift=-12)                 # incomes have fallen
for col in X_train.columns:
    ks = stats.ks_2samp(X_train[col], X_live[col])
    print(f"{col:6s}  PSI {psi(X_train[col], X_live[col]):.3f}   KS statistic {ks.statistic:.3f}   p-value {ks.pvalue:.3g}")

income  PSI 0.612   KS statistic 0.312   p-value 1.12e-123
debt    PSI 0.003   KS statistic 0.021   p-value 0.536
age     PSI 0.007   KS statistic 0.015   p-value 0.892


## 7. Concept drift and retraining

In [4]:
# 12 months of new loans; from month 5 the effect of income fades and then reverses (concept drift).
static, retrained, current = [], [], model
for month in range(1, 13):
    flip = 0.0 if month < 5 else 0.02 * (month - 4)
    Xm, ym = make_loans(1500, flip=flip)
    static.append(roc_auc_score(ym, model.predict_proba(Xm)[:, 1]))
    retrained.append(roc_auc_score(ym, current.predict_proba(Xm)[:, 1]))
    current = make_pipeline(StandardScaler(), LogisticRegression()).fit(Xm, ym)   # retrain on this month's labels
print("AUC, never retrained:   ", [round(a, 2) for a in static])
print("AUC, retrained monthly: ", [round(a, 2) for a in retrained])

AUC, never retrained:    [0.82, 0.83, 0.82, 0.84, 0.77, 0.76, 0.71, 0.65, 0.58, 0.55, 0.46, 0.41]
AUC, retrained monthly:  [0.82, 0.83, 0.82, 0.84, 0.77, 0.77, 0.74, 0.72, 0.73, 0.79, 0.78, 0.83]


## 8. Testing ML code

In [5]:
def report(name, ok):
    print(("PASS " if ok else "FAIL ") + name)
base = pd.DataFrame({"income": [50.0], "debt": [20.0], "age": [40]})
p0 = model.predict_proba(base)[0, 1]
report("minimum quality: test AUC above 0.70", meta["test_auc"] > 0.70)
report("directional: more debt means higher risk", model.predict_proba(base.assign(debt=35.0))[0, 1] > p0)
report("directional: more income means lower risk", model.predict_proba(base.assign(income=80.0))[0, 1] < p0)
report("invariance: age changes risk by under 2 points", abs(model.predict_proba(base.assign(age=65))[0, 1] - p0) < 0.02)
report("output is a valid probability", 0 <= p0 <= 1)

PASS minimum quality: test AUC above 0.70
PASS directional: more debt means higher risk
PASS directional: more income means lower risk
PASS invariance: age changes risk by under 2 points
PASS output is a valid probability


## 10. In Python: a prediction service function

In [6]:
import json, time
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

df = pd.read_csv("../data/houses.csv")   # download it from Lesson 1
FEATURES = ["size_m2", "bedrooms", "dist_km", "age_yrs"]
model = make_pipeline(StandardScaler(), Ridge(1.0)).fit(df[FEATURES], df["price_lakh"])
VERSION, LOG = "houses-1.0.0", []

def handle(request_json):
    req = json.loads(request_json)
    missing = [f for f in FEATURES if not isinstance(req.get(f), (int, float))]
    if missing:
        result = {"error": "missing or invalid: " + ", ".join(missing), "model": VERSION}
    else:
        price = model.predict(pd.DataFrame([req])[FEATURES])[0]
        result = {"price_lakh": round(float(price), 1), "model": VERSION}
    LOG.append({"time": time.time(), "request": req, "response": result})
    return json.dumps(result)

print(handle('{"size_m2": 110, "bedrooms": 3, "dist_km": 6, "age_yrs": 4}'))
print(handle('{"size_m2": 110, "bedrooms": "three", "dist_km": 6}'))
print("requests logged:", len(LOG))

{"price_lakh": 91.9, "model": "houses-1.0.0"}
{"error": "missing or invalid: bedrooms, age_yrs", "model": "houses-1.0.0"}
requests logged: 2


## 13. Going deeper: drift alarms and champion–challenger tests

In [7]:
import numpy as np
from scipy import stats
rng = np.random.default_rng(1)
train = rng.normal(0, 1, 50000)
for n in [500, 5000, 50000]:
    live = rng.normal(0.03, 1, n)                                    # a tiny, harmless shift
    ks = stats.ks_2samp(train, live)
    edges = np.quantile(train, np.linspace(0, 1, 11)); edges[0], edges[-1] = -np.inf, np.inf
    e, a = np.histogram(train, edges)[0] / len(train), np.histogram(live, edges)[0] / n
    psi = float(np.sum((a - e) * np.log(a / e)))
    print(f"live sample {n:6d}: KS p-value {ks.pvalue:.3g}   PSI {psi:.4f}")

live sample    500: KS p-value 0.158   PSI 0.0289
live sample   5000: KS p-value 0.0375   PSI 0.0031
live sample  50000: KS p-value 1.08e-06   PSI 0.0021


In [8]:
from sklearn.metrics import roc_auc_score
m = 3000
y = rng.integers(0, 2, m)
champion = y * 1.0 + rng.normal(0, 1.2, m)                          # current model's scores
challenger = y * 1.0 + rng.normal(0, 1.1, m)                        # new model, slightly better
diffs = []
for _ in range(1000):                                               # bootstrap the AUC difference
    idx = rng.integers(0, m, m)
    diffs.append(roc_auc_score(y[idx], challenger[idx]) - roc_auc_score(y[idx], champion[idx]))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print("AUC champion", round(roc_auc_score(y, champion), 3), " challenger", round(roc_auc_score(y, challenger), 3))
print("95% interval for the improvement:", round(lo, 3), "to", round(hi, 3))

AUC champion 0.725  challenger 0.728
95% interval for the improvement: -0.022 to 0.027


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [9]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
cols = ["size_m2", "bedrooms", "dist_km", "age_yrs"]
pipe = make_pipeline(StandardScaler(), Ridge(1.0)).fit(df[cols], df.price_lakh)
joblib.dump({"model": pipe, "features": cols, "version": "1.0.0"}, "price_model.joblib")
bundle = joblib.load("price_model.joblib")
new = pd.DataFrame({"size_m2": [95], "bedrooms": [2], "dist_km": [5], "age_yrs": [3]})
print("model", bundle["version"], "predicts", round(float(bundle["model"].predict(new[bundle["features"]])[0]), 1), "lakh")
ranges = {c: (df[c].min(), df[c].max()) for c in cols}
print("inside the training range:", {c: bool(ranges[c][0] <= new[c].iloc[0] <= ranges[c][1]) for c in cols})

model 1.0.0 predicts 78.9 lakh
inside the training range: {'size_m2': True, 'bedrooms': True, 'dist_km': True, 'age_yrs': True}


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [10]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Population stability index

Compute the PSI between expected and actual bin shares.

In [11]:
import numpy as np
e = np.array([0.25, 0.25, 0.25, 0.25])
a = np.array([0.10, 0.20, 0.30, 0.40])
psi = np.sum(a - e)  # ← multiply by the log ratio
result = round(float(psi), 4)
check(result, 0.2282, hint='Sum (a − e) × ln(a ÷ e) over the bins.')

❌ Not yet. Hint: Sum (a − e) × ln(a ÷ e) over the bins.


### Challenge 2: Validate a row

Count the fields that are missing or not numbers.

In [12]:
row = {"income": 50.0, "debt": None, "age": "40"}
fields = ["income", "debt", "age"]
bad = 0  # ← count the fields that fail the check
check(bad, 2, hint="Check each field's type with isinstance and count the failures.")

❌ Not yet. Hint: Check each field's type with isinstance and count the failures.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Population stability index** — the missing line:

```python
psi = np.sum((a - e) * np.log(a / e))
```

**2. Validate a row** — the missing line:

```python
bad = sum(1 for f in fields if not isinstance(row.get(f), (int, float)))
```